In [75]:
import polars as pl
import numpy as np
import geopandas as gpd
from pathlib import Path
import pyarrow
import matplotlib.pyplot as plt
from IPython.display import Image

%matplotlib inline

PARENT_FOLDER = Path().resolve().parent
CURRENT_DIR = Path().resolve()

In [ ]:
file_path = CURRENT_DIR.parent / "Data" / "Travel" / "US Commute Flows by Residence County.csv"
us_commute_flows = pl.read_csv(file_path, encoding="utf8")

file_path = CURRENT_DIR.parent / "Data" / "Travel" / "UK Commute Flows by Area Code.csv"
uk_commute_flows = pl.read_csv(file_path, encoding="utf8")

In [95]:
uk_commute_flows

Residence_Area_Code,Residence_Area_Name,Workplace_Area_Code,Workplace_Area_Name,Count
str,str,str,str,i32
"""E06000001""","""Hartlepool""","""E06000001""","""Hartlepool""",14513
"""E06000001""","""Hartlepool""","""E06000002""","""Middlesbrough""",1500
"""E06000001""","""Hartlepool""","""E06000003""","""Redcar and Cleveland""",671
"""E06000001""","""Hartlepool""","""E06000004""","""Stockton-on-Tees""",4240
"""E06000001""","""Hartlepool""","""E06000005""","""Darlington""",398
…,…,…,…,…
"""W06000024""","""Merthyr Tydfil""","""W06000020""","""Torfaen""",228
"""W06000024""","""Merthyr Tydfil""","""W06000021""","""Monmouthshire""",254
"""W06000024""","""Merthyr Tydfil""","""W06000022""","""Newport""",242


In [ ]:
uk_commute_flows = uk_commute_flows.with_columns(
    pl.col("Count").str.replace_all(r"[^0-9]", "").cast(pl.Int32).alias("Count")
)
uk_commute_flows

us_commute_flows = us_commute_flows.with_columns(
    pl.col("Workers_In_Commuting_Flow").str.replace_all(r"[^0-9]", "").cast(pl.Int32),
    pl.col("Margin_of_Error").str.replace_all(r"[^0-9]", "").cast(pl.Int32),
)

In [ ]:
uk_commute_flows.group_by("Workplace_Area_Name").agg(pl.col("Count").sum().alias("Total_Trips_To_Workplace")).sort(
    "Total_Trips_To_Workplace", descending=True
)

Workplace_Area_Name,Total_Trips_To_Workplace
str,i32
"""Birmingham""",287033
"""Leeds""",222151
"""Westminster""",199443
"""Manchester""",184962
"""Sheffield""",153589
…,…
"""Mid Ulster""",37
"""East Dunbartonshire""",35
"""Clackmannanshire""",34


In [ ]:
us_commute_flows.group_by("Workplace_County_Name").agg(pl.col("Workers_In_Commuting_Flow").sum()).sort(
    "Workers_In_Commuting_Flow", descending=True
)

Workplace_County_Name,Workers_In_Commuting_Flow
str,i32
"""Los Angeles County""",4906906
"""Orange County""",2806483
"""Cook County""",2684000
"""Harris County""",2520499
"""New York County""",2494885
…,…
"""Treasure County""",234
"""Petroleum County""",229
"""Issaquena County""",203


In [ ]:
import polars as pl
import geopandas as gpd
import folium
from pathlib import Path

# --- Configuration ---
# Focus on one core city to build its Metropolitan Area
CORE_CITY = "Manchester"
# Threshold: If >15% of a district's workers commute to Core, it's part of the Metro area
COMMUTER_THRESHOLD = 0.15

# URL for UK Local Authority District Boundaries (GeoJSON) - Open Geography Portal
# Using generalized boundaries for faster rendering
GEOJSON_URL = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_May_2023_Boundaries_UK_BGC/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"

CURRENT_DIR = Path().resolve()


def load_and_clean_data(filepath):
    """
    Loads commute data and cleans numeric columns using Polars.
    """
    return (
        pl.scan_csv(filepath, encoding="utf8")
        .with_columns(pl.col("Count").str.replace_all(r"[^0-9]", "").cast(pl.Int32))
        .collect()
    )


def calculate_commuter_metrics(df, core_city):
    """
    Calculates the percentage of resident workers in every district
    who commute to the core_city.
    """
    # 1. Calculate Total Resident Workers per District (Sum of all outflows)
    resident_totals = df.group_by("Residence_Area_Code").agg(pl.col("Count").sum().alias("Total_Resident_Workers"))

    # 2. Extract flows specifically to the Core City
    flows_to_core = (
        df.filter(pl.col("Workplace_Area_Name") == core_city)
        .select(["Residence_Area_Code", "Residence_Area_Name", "Count"])
        .rename({"Count": "Commuters_To_Core"})
    )

    # 3. Join and Calculate Percentage
    metrics = (
        flows_to_core.join(resident_totals, on="Residence_Area_Code", how="left")
        .with_columns((pl.col("Commuters_To_Core") / pl.col("Total_Resident_Workers")).alias("Commuter_Pct"))
        .sort("Commuter_Pct", descending=True)
    )

    return metrics


# --- Execution ---

# 1. Load Data
# Assuming file is in the relative path defined in original notebook
file_path = CURRENT_DIR.parent / "Data" / "Travel" / "UK Commute Flows by Area Code.csv"
uk_commute_flows = load_and_clean_data(file_path)

# 2. Calculate Metrics for the Core City
metro_data = calculate_commuter_metrics(uk_commute_flows, CORE_CITY)

print(f"Top districts connected to {CORE_CITY} by commuter flow:")
print(metro_data.head(10))

# 3. Prepare Geometry for Mapping
print("Loading geometry (this may take a moment)...")
gdf_boundaries = gpd.read_file(GEOJSON_URL)

# Merge Census metrics with Geometry
# Note: Ensure the GeoJSON column for code (e.g., LAD23CD) matches 'Residence_Area_Code'
# We rename columns to match standardized ONS codes usually found in these GeoJSONs
gdf_boundaries = gdf_boundaries.rename(columns={"LAD23CD": "Residence_Area_Code", "LAD23NM": "Residence_Area_Name"})

# Convert Polars DataFrame to Pandas for merging with GeoPandas
metro_metrics_pd = metro_data.to_pandas()
map_data = gdf_boundaries.merge(metro_metrics_pd, on="Residence_Area_Code", how="left")

# Fill NaN for areas with no flow (0 commuters)
map_data["Commuter_Pct"] = map_data["Commuter_Pct"].fillna(0)

# Define the Metropolitan Area (The Core + Commuter Zone)
map_data["Category"] = "Outside Metro Area"
map_data.loc[map_data["Residence_Area_Name_x"] == CORE_CITY, "Category"] = "Urban Core"
map_data.loc[
    (map_data["Commuter_Pct"] >= COMMUTER_THRESHOLD) & (map_data["Residence_Area_Name_x"] != CORE_CITY), "Category"
] = "Commuter Zone"

# --- Visualization ---

# Create a map centered on the Core
core_centroid = map_data[map_data["Residence_Area_Name_x"] == CORE_CITY].geometry.centroid.iloc[0]
m = folium.Map(location=[core_centroid.y, core_centroid.x], zoom_start=9, tiles="cartodbpositron")


# Function to style the polygons
def style_function(feature):
    category = feature["properties"]["Category"]
    return {
        "fillColor": "#ff0000" if category == "Urban Core" else "#ff9900" if category == "Commuter Zone" else "#eeeeee",
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.6 if category != "Outside Metro Area" else 0.1,
    }


# Add the layer
folium.GeoJson(
    map_data,
    name="Metropolitan Area",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["Residence_Area_Name_x", "Commuter_Pct", "Category"],
        aliases=["District:", "Commuter %:", "Type:"],
        localize=True,
    ),
).add_to(m)

# Add a legend or control
folium.LayerControl().add_to(m)

# Display map
m

FileNotFoundError: The system cannot find the path specified. (os error 3): D:\Public Transport Study\Data\Travel\UK Commute Flows by Area Code.csv

This error occurred with the following context stack:
	[1] 'csv scan'
	[2] 'with_columns'
